# Chebyshev center

Find the center and radius of the largest sphere that fits inside the polyhedron defined by the inequalities
$$ \left\{ 2 x_1 - x_2 + 2 x_3 \le 2,\quad
-x_1 + 2 x_2 + 4 x_3 \le 16,\quad
x_1 + 2 x_2 - 2 x_3 \le 8,\quad
x_1 \ge 0,\,
x_2 \ge 0,\,
x_3 \ge 0 \right\}$$

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and choose **Run All**. JuMP, HiGHS, and Plots are already included; LinearAlgebra is part of Julia. No external data files are needed.


## Optimization problem formulation

The Chebyshev center problem can be formulated as the following linear program:

$\max_{x,r} ~~r $

subject to

$ a_j^T x + \|a_j\|_2 r\leq b_j\quad$ for $j=1,...,6$, with $r \geq 0$. 

Here $a_j$ is the nonzero normal vector of hyperplane $j$, $\|a_j\|_2$ is its Euclidean norm, and $b_j$ is its right-hand side.

$a_1 = [ 2~~-\!1~~2]$,

$a_2 = [ -\!1~~2~~4]$, 

$a_3 = [ 1~~2~~-\!2]$, 

$a_4 = [ -\!1~~0~~0]$, 

$a_5 = [ 0~~-\!1~~0]$, 

$a_6 = [ 0~~0~~-\!1]$, and

$b = [2~~16~~8~~0~~0~~0]$.

A ball with center $x$ and radius $r$ consists of points $x+u$ with $\|u\|_2 \le r$. The largest value of $a_j^T(x+u)$ over this ball is $a_j^T x + r\|a_j\|_2$, so each constraint ensures that the entire ball lies on the feasible side of a boundary plane. The norms are constants computed from the data, making this a linear program.

The last three rows encode the coordinate boundaries. They require $x_1 \ge r$, $x_2 \ge r$, and $x_3 \ge r$: requiring only a nonnegative center would not keep the whole ball inside the nonnegative orthant.


## JuMP implementation

In [ ]:
# Matrices A, b defining the inequalities (each row vector is a normal vector to a hyperplane!)
A = [2 -1 2; -1 2 4; 1 2 -2; -1 0 0; 0 -1 0; 0 0 -1];
b = [2; 16; 8; 0; 0; 0]

using JuMP, HiGHS, LinearAlgebra
import MathOptInterface as MOI

m = Model(HiGHS.Optimizer)
set_silent(m)  # Remove this line to see the solver log.
@variable(m, r >= 0)           # radius of the sphere
@variable(m, x[1:3])          # coordinates of center; boundary rows impose x[i] >= r
for i = 1:size(A,1) # number of rows
    @constraint(m, A[i,:]'*x + r*norm(A[i,:]) <= b[i]) # Euclidean norm of the row normal.
end
@objective(m, Max, r)     # maximize radius of the sphere

optimize!(m)
if termination_status(m) != MOI.OPTIMAL || primal_status(m) != MOI.FEASIBLE_POINT
    error("No optimal primal solution: $(termination_status(m)), $(primal_status(m))")
end
center = value.(x)
radius = value(r)

println("HiGHS LP solver terminated with status ", termination_status(m))
println("The coordinates of the Chebyshev center are: ", center)
println("The largest possible radius is: ", radius)

## Visualizing the solution

The wireframe shows the feasible polyhedron $Ax \leq b$. The blue sphere has the optimized `center` and `radius`, and the red point marks its center. Equal axis scales preserve the geometry.

For this bounded three-dimensional example, we find the vertices by intersecting triples of boundary planes and keeping only feasible intersections. Two vertices form an edge when they share two independent active boundary planes. The plotting code uses `Plots`, which is already included in the course environment.


In [ ]:
# Recover vertices and edges from the original inequalities Ax <= b.
tol = 1e-8
vertices = Vector{Float64}[]
number_of_planes = size(A, 1)
for i in 1:(number_of_planes-2), j in (i+1):(number_of_planes-1),
    k in (j+1):number_of_planes
    rows = [i, j, k]
    normals = Float64.(A[rows, :])
    rank(normals) == 3 || continue  # Skip nonunique intersections.
    vertex = normals \ b[rows]
    if all(A * vertex .<= b .+ tol) &&
       all(norm(vertex - v) > tol for v in vertices)
        push!(vertices, vertex)
    end
end

edges = Tuple{Int,Int}[]
for i in 1:(length(vertices)-1), j in (i+1):length(vertices)
    active_i = abs.(A * vertices[i] - b) .<= tol
    active_j = abs.(A * vertices[j] - b) .<= tol
    shared = findall(active_i .& active_j)
    if length(shared) >= 2 && rank(Float64.(A[shared, :])) == 2
        push!(edges, (i, j))
    end
end


In [ ]:
using Plots
gr()

# Parameterize the sphere using azimuth theta and polar angle phi.
theta = range(0, 2pi; length=81)
phi = range(0, pi; length=41)
sphere_x = [center[1] + radius * cos(t) * sin(p) for t in theta, p in phi]
sphere_y = [center[2] + radius * sin(t) * sin(p) for t in theta, p in phi]
sphere_z = [center[3] + radius * cos(p) for t in theta, p in phi]

# Use identical limits on all axes so one unit has the same scale.
coordinates = hcat(vertices...)
lo, hi = extrema(coordinates)
padding = 0.08 * (hi - lo)
limits = (lo - padding, hi + padding)
chebyshev_plot = surface(
    sphere_x, sphere_y, sphere_z;
    color=:steelblue, colorbar=false, label="",
    xlabel="x1", ylabel="x2", zlabel="x3",
    xlims=limits, ylims=limits, zlims=limits, aspect_ratio=:equal,
    camera=(35, 25), size=(800, 650), legend=:topright,
    title="Chebyshev center: radius = $(round(radius; digits=3))")
for (edge_number, (i, j)) in enumerate(edges)
    ends = hcat(vertices[i], vertices[j])
    plot!(chebyshev_plot, ends[1, :], ends[2, :], ends[3, :];
        color=:black, linewidth=2,
        label=edge_number == 1 ? "Polyhedron" : "")
end
scatter!(chebyshev_plot, [center[1]], [center[2]], [center[3]];
    color=:red, markersize=5, label="Center")
chebyshev_plot


### Three more viewing angles

Each view copies the original plot and changes only `camera=(azimuth, elevation)`, measured in degrees. The solution, geometry, and axis scales stay the same. Adjust these angles and rerun a cell to explore another viewpoint.


In [ ]:
side_view = deepcopy(chebyshev_plot)
plot!(side_view; camera=(120, 30),
    title="View from another side")


In [ ]:
high_view = deepcopy(chebyshev_plot)
plot!(high_view; camera=(35, 70),
    title="View from above")


In [ ]:
low_view = deepcopy(chebyshev_plot)
plot!(low_view; camera=(35, 10),
    title="View near ground level")
